<!--
Copyright 2026 Google LLC

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

    http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.
-->

# **Torch.compile on TorchTPU**

This guide provides best practices for using `torch.compile` to accelerate PyTorch models on Google TPUs.

In [1]:
import sys
import torch

# One time initialization of the TPU
device = torch.device("tpu")

Device type: tpu, Device index: default
Initializing TPU distributed runtime


I0000 00:00:1787340622.018493   75955 pjrt_api.cc:119] GetPjrtApi was found for TPU at /home/mateusz/miniforge3/envs/tpu-experiment/lib/python3.12/site-packages/libtpu/libtpu.so
I0000 00:00:1787340622.035174   75955 tpu_profiler_plugin.cc:791] PyTorch PrivateUse1ProfilerRegistry not found. Native TPU profiling will be disabled (this is expected on PyTorch < 2.12).
Successfully renamed PrivateUse1 backend to 'tpu'. Device: device(type='tpu')
Registered Python module for 'tpu'.
I0000 00:00:1787340622.056608   75955 device_rt.cc:130] PjRt runtime initialization deferred for tpu


## **1. The `dynamic=False` Requirement**

As of today, `torch_tpu` requires static shapes when using `torch.compile`. We do not currently support the `dynamic=True` or `dynamic=None` flags.

**What this means for you:**
You must always set `dynamic=False`. As a consequence, if you pass inputs with variable dimensions (such as a changing batch size at the end of an epoch, or variable sequence lengths), this will trigger a full recompilation every time the size changes. Recompilations are expensive and will reduce your training performance.

**Best Practice:**
Do not pass variable sizes to your compiled models. Ensure your tensor dimensions remain constant across all iterations. If your dataset naturally has varying sizes, use standard PyTorch techniques to **pad** or **slice** the data *outside* of the compiled region to maintain fixed-size input tensors.

In [2]:
class MyLargeModel(torch.nn.Module):

  def __init__(self):
    super().__init__()
    self.net = torch.nn.Sequential(
        torch.nn.Linear(10, 128), torch.nn.ReLU(), torch.nn.Linear(128, 10)
    )

  def forward(self, x):
    return self.net(x)

# Best Practice: Model and data are allocated directly on the hardware
with torch.device("tpu"):
  model = MyLargeModel()
  input_data = torch.randn(20, 10)

# Initialize the compiled model with mandatory static shapes
compiled_model = torch.compile(model, backend="tpu", dynamic=False)

print(
    "Starting TPU XLA compilation (this may take 1-2 minutes)...", flush=True
)
# Compilation Warming: The first forward pass triggers the actual TPU compilation
warmup_output = compiled_model(input_data)
print("Compilation complete!", flush=True)

print(f"Compilation warmup complete! Output shape: {warmup_output.shape}")

I0000 00:00:1787340622.064073   75955 pjrt_api.cc:168] The PJRT plugin has PJRT API version 0.114. The framework PJRT API version is 0.114.


Starting TPU XLA compilation (this may take 1-2 minutes)...


I0000 00:00:1787340623.452218   75955 pjrt_c_api_client.cc:188] PjRtCApiClient created.
I0000 00:00:1787340623.470367   76190 compilation_cache_utils.cc:217] Tier-3 compilation cache root directory: 
I0000 00:00:1787340623.470408   76190 tier2_compilation_cache.cc:82] Tier-2 compilation cache is disabled for world size 1.
I0000 00:00:1787340623.470411   76190 tier3_compilation_cache.cc:50] Backup compilation for tier-3 cache read is disabled as tier-3 cache is disabled.


class GraphModule(torch.nn.Module):
    def forward(self, primals_1: "f32[128, 10]", primals_2: "f32[128]", primals_3: "f32[20, 10]", primals_4: "f32[10, 128]", primals_5: "f32[10]"):
        # No stacktrace found for following nodes
        submod_0 = self.submod_0(primals_3, primals_1, primals_2, primals_4, primals_5);  primals_3 = primals_1 = primals_2 = primals_4 = primals_5 = None
        getitem = submod_0[0]
        getitem_1 = submod_0[1]
        getitem_2 = submod_0[2]
        getitem_3 = submod_0[3]
        getitem_4 = submod_0[4];  submod_0 = None
        return (getitem, getitem_1, getitem_2, getitem_3, getitem_4)

    class submod_0(torch.nn.Module):
        def forward(self, primals_3: "f32[20, 10]", primals_1: "f32[128, 10]", primals_2: "f32[128]", primals_4: "f32[10, 128]", primals_5: "f32[10]"):
            # No stacktrace found for following nodes
            clone_default: "f32[20, 10]" = torch.ops.aten.clone.default(primals_3);  primals_3 = None

            # File:

Compilation warmup complete! Output shape: torch.Size([20, 10])


/home/mateusz/miniforge3/envs/tpu-experiment/lib/python3.12/site-packages/torch_tpu/_internal/export/export.py:533: UserWarning: 'default_generators' is experimental and subject to change: Default generators for TPU devices.
  ] + [device_module.default_generators[device.index or 0]]
